### Modelo Baseline (Regressão Logística)

#### 1. Configuração do ambiente

In [ ]:
import logging

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

RANDOM_STATE = 7
TEST_SIZE = 0.2

#### 2. Carregamento dos dados pré-processados

In [2]:
df = pd.read_csv("../data/telco_customer_churn_preprocessed.csv")

#### 3. Criação de modelo baseline

In [ ]:
num_features = ["tenure_months", "monthly_charges", "total_charges", "cltv"]
cat_features = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
]

# Preparação dos dados
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Pré-processamento
num_transformer = make_pipeline([("scaler", StandardScaler())])
cat_transformer = make_pipeline([("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", drop="first"))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
    ]
)

# Integração do Pipeline, definição do modelo
pipeline = make_pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(random_state=RANDOM_STATE))])

# Treino e avaliação
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

logger.info("\n=== LOGISTIC REGRESSION ===")
logger.info(classification_report(y_test, y_pred))

INFO: 
=== LOGISTIC REGRESSION ===


INFO:               precision    recall  f1-score   support

           0       0.86      0.89      0.88      1035
           1       0.67      0.60      0.63       374

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.81      0.81      0.81      1409



#### 4. Persistência do modelo baseline

In [19]:
joblib.dump(pipeline, "../models/baseline_logistic_regression_pipeline.joblib")

['../models/baseline_logistic_regression_pipeline.joblib']